# Plantilla de algoritmo. Parte 3 (Problema del Viajante)

**Cómo usarla:** copia este archivo como `parte3/<algoritmo>_<tunombre>.ipynb` (por ejemplo, `parte3/fuerza_bruta_pablo.ipynb`) y trabaja solo **debajo de la línea ✂️**. Lo de arriba (preparación) es el mismo código de la Parte 2 del trabajo y sirve para cargar las versiones del grafo: **no lo modifiques**. La guía completa está en `CONTRIBUTING.md`.

> ⚙️ El notebook se ejecuta desde su carpeta (`parte3/`): la primera celda sube a la raíz del proyecto para que las rutas (`data/...`) funcionen.

In [1]:
# Sube a la raíz del proyecto para que las rutas (data/ciudades.csv, data/versiones/...) funcionen
import os
from pathlib import Path
if Path.cwd().name == "parte3":
    os.chdir("..")

In [2]:
# Librerías que se usan en todo el notebook
import csv                       # leer y escribir tablas en formato CSV
import itertools                 # recorrer todas las combinaciones de parámetros
import unicodedata               # quitar acentos de los nombres (Cataluña -> cataluna)
from pathlib import Path         # rutas de archivos

import networkx as nx            # crear y analizar grafos (la misma librería de las prácticas)
import numpy as np               # tablas de números (las tablas de distancias)
from scipy.sparse import csr_matrix                 # matriz del grafo para el algoritmo de Dijkstra
from scipy.sparse.csgraph import dijkstra           # caminos mínimos desde varios orígenes
from scipy.spatial import cKDTree                   # buscar rápido el nodo más cercano a un punto

from tsp_utils import dibujar_red  # función de dibujo de la red de carreteras, comentada en tsp_utils.py

In [3]:
# ── Parámetros de las versiones (editables) ──
X_KM  = 10             # una ciudad a más de X_KM km de la red se deja fuera de la versión
N_MIN = 4              # mínimo de ciudades para una versión por comunidad autónoma
RECALCULAR = False     # True: vuelve a calcular las 16 tablas (necesita el grafo G de la Parte 1)
CARPETA = Path("data/versiones")     # aquí se guardan las tablas ya calculadas

# Qué tipos de vía (`highway`) puede usar cada versión de carreteras
CARRETERAS = {
    "completo":             ["motorway", "motorway_link", "trunk", "trunk_link",
                             "primary", "primary_link", "secondary", "secondary_link"],
    "autovias_principales": ["motorway", "motorway_link", "trunk", "trunk_link",
                             "primary", "primary_link"],
    "solo_autovias":        ["motorway", "motorway_link"],
    "sin_autovias":         ["trunk", "trunk_link", "primary", "primary_link",
                             "secondary", "secondary_link"],
}
PESOS = {"km": ("length", 1000), "tiempo": ("travel_time", 1)}   # peso -> (atributo de la arista, divisor)
CONJUNTOS = ["capitales", "capitales_y_100k"]                    # qué ciudades se visitan
GENERALES = ["multigrafo_dirigido", "grafo_dirigido",
             "multigrafo_no_dirigido", "grafo_simple_no_dirigido"]

In [4]:
# Lee la lista de ciudades (nombre, comunidad, si es capital de provincia, coordenadas...)
with open("data/ciudades.csv", encoding="utf-8") as f:
    CIUDADES = list(csv.DictReader(f))
for c in CIUDADES:
    c["lat"], c["lon"] = float(c["lat"]), float(c["lon"])
    c["capital"] = c["capital_provincia"] == "True"

def slug(texto):
    # "Castilla y León" -> "castilla_y_leon" (sin acentos ni espacios, para usarlo como nombre)
    sin_acentos = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    return sin_acentos.lower().replace(" ", "_").replace("-", "_")

AMBITOS = ["espana"] + sorted({slug(c["comunidad"]) for c in CIUDADES})

def ciudades_de(nodos, ambito):
    # Nombres de las ciudades de un conjunto: nodos = "capitales" o "capitales_y_100k";
    # ambito = "espana" o una comunidad (p. ej. "galicia")
    return [c["nombre"] for c in CIUDADES
            if (c["capital"] or nodos == "capitales_y_100k")
            and (ambito == "espana" or slug(c["comunidad"]) == ambito)]

print(f"{len(CIUDADES)} ciudades | {len(ciudades_de('capitales', 'espana'))} capitales | ámbitos: {len(AMBITOS) - 1} comunidades")
print("Capitales de Galicia:", ciudades_de("capitales", "galicia"))

75 ciudades | 47 capitales | ámbitos: 15 comunidades
Capitales de Galicia: ['A Coruña', 'Ourense', 'Lugo', 'Pontevedra']


In [5]:
def nombre_tabla(aristas, peso, dirigido):
    return CARPETA / f"{aristas}__{peso}__{'dirigido' if dirigido else 'no_dirigido'}.csv"

def guardar_tabla(D, nombres, ruta):
    with open(ruta, "w", encoding="utf-8") as f:
        np.savetxt(f, D, delimiter=",", header=",".join(nombres), comments="", fmt="%.3f")

def leer_tabla(aristas, peso, dirigido):
    ruta = nombre_tabla(aristas, peso, dirigido)
    with open(ruta, encoding="utf-8") as f:
        nombres = f.readline().strip().split(",")        # primera fila: nombres de las ciudades
    return np.loadtxt(ruta, delimiter=",", skiprows=1, encoding="utf-8"), nombres

def escribir_csv(ruta, filas, campos):
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=campos)
        w.writeheader()
        w.writerows(filas)

In [6]:
UNIDAD = {"km": "km", "tiempo": "segundos"}

def cargar_version(aristas, nodos, ambito, peso, general):
    # aristas: "completo" | "autovias_principales" | "solo_autovias" | "sin_autovias"
    # nodos:   "capitales" | "capitales_y_100k"       ambito: "espana" | "galicia" | "andalucia" ...
    # peso:    "km" | "tiempo"                        general: uno de GENERALES
    # (multigrafo y grafo simple dan el mismo grafo de ciudades: no hay aristas repetidas)
    dirigido = "no_dirigido" not in general
    D, nombres = leer_tabla(aristas, peso, dirigido)
    elegidas = [c for c in ciudades_de(nodos, ambito) if c in nombres]    # sin las que quedaron fuera
    if ambito != "espana" and len(elegidas) < N_MIN:
        raise ValueError(f"Solo hay {len(elegidas)} ciudades (mínimo {N_MIN}) en {ambito}")
    K = nx.DiGraph() if dirigido else nx.Graph()
    for c in CIUDADES:
        if c["nombre"] in elegidas:
            K.add_node(c["nombre"], lat=c["lat"], lon=c["lon"])
    for p, a in enumerate(elegidas):
        for q, b in enumerate(elegidas):
            if p != q and (dirigido or p < q):                            # no dirigido: una arista por par
                K.add_edge(a, b, peso=float(D[nombres.index(a), nombres.index(b)]))
    K.graph.update(aristas=aristas, nodos=nodos, ambito=ambito, peso=peso,
                   unidad=UNIDAD[peso], general=general,
                   fuera=[c for c in ciudades_de(nodos, ambito) if c not in nombres])
    return K

In [7]:
# [AYUDA COMPARTIDA] Funciones comunes a todos los algoritmos
import time
import tracemalloc

def coste_ruta(K, ruta):
    # Coste total del ciclo: suma de las distancias entre ciudades consecutivas y de la última a la primera
    return sum(K[ruta[i]][ruta[(i + 1) % len(ruta)]]["peso"] for i in range(len(ruta)))

def validar_ruta(K, ruta):
    # Comprueba que la ruta visita cada ciudad de K exactamente una vez
    assert len(ruta) == K.number_of_nodes() and set(ruta) == set(K.nodes), "la ruta no visita cada ciudad una vez"

def medir(algoritmo, K, memoria=False, **opciones):
    # Ejecuta algoritmo(K, **opciones), comprueba el resultado y devuelve cuánto ha tardado
    # (con memoria=True mide también el pico de memoria en una segunda ejecución, porque medirla ralentiza)
    t0 = time.perf_counter()
    ruta, coste = algoritmo(K, **opciones)
    segundos = time.perf_counter() - t0
    validar_ruta(K, ruta)
    assert abs(coste - coste_ruta(K, ruta)) < 1e-6 * max(1.0, abs(coste)), "el coste devuelto no coincide con el de la ruta"
    resultado = {"algoritmo": algoritmo.__name__, "ciudades": K.number_of_nodes(), "coste": round(coste, 3),
                 "unidad": K.graph.get("unidad"), "segundos": round(segundos, 4)}
    if memoria:
        tracemalloc.start()
        algoritmo(K, **opciones)
        _, pico = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        resultado["memoria_mb"] = round(pico / 1e6, 2)
    return resultado

---
### ✂️ A PARTIR DE AQUÍ SE INTEGRA EN EL TRABAJO

Todo lo que escribas **debajo de esta línea** pasará al notebook principal (`trabajo_tsp.ipynb`, Parte 3). No modifiques esta celda.

### 2-opt (mejora local)

**🗂️ FICHA**

| | |
|---|---|
| **Autor** | Raimundo |
| **Función** | `dos_opt(K, inicio=None, ruta_inicial=None)` |
| **Tipo** | Heurístico de mejora local |
| **Determinista** | Sí |
| **Garantía de calidad** | Ninguna |
| **Distancias simétricas** | No la necesita |
| **Complejidad en tiempo** | O(n²) por vuelta (≈ O(n³) en la práctica) |
| **Complejidad en memoria** | O(n²) |
| **Límite práctico** | Todas |
| **Uso de IA** | Texto y código elaborados con ayuda de Claude (Anthropic); probados y revisados por el autor. |

<div style="border-left:5px solid #4C72B0;color:#000;background:#EEF2FA;padding:10px 14px;margin:10px 0;border-radius:4px;">

**📚 TEORÍA**

Es una **búsqueda local**: parte de un recorrido cualquiera y lo va **mejorando con pequeños cambios** mientras se pueda. El cambio de 2-opt es el más sencillo que se le ocurriría a una persona: si el recorrido tiene **dos aristas que se cruzan**, se desharán y se volverán a unir "al revés", lo que siempre elimina el cruce y suele acortar el camino.

**Idea.** Se toman dos aristas del ciclo, `(a → b)` y `(c → d)` (con `b` justo después de `a` y `d` justo después de `c`), y se sustituyen por `(a → c)` y `(b → d)`. Para que el resultado siga siendo un único ciclo, **se invierte el tramo** `b … c`. Ese cambio solo merece la pena si

> `coste(a→c) + coste(b→d) < coste(a→b) + coste(c→d)`

(en un grafo **dirigido** hay que sumar además la diferencia entre recorrer el tramo invertido al revés y hacia delante, porque ir de `x` a `y` no cuesta lo mismo que de `y` a `x`; el código lo tiene en cuenta con sumas acumuladas, y vale igual para el no dirigido, donde esa diferencia es cero). Se repite hasta que **ningún par de aristas permita mejorar**: entonces la ruta es un **óptimo local** ("2-óptima").

**Pseudocódigo.**

<pre style="background:#fff;border:1px solid #ccd;padding:10px 14px;border-radius:4px;font-size:13px;">
DOS_OPT(K, inicio, ruta_inicial):
    ruta ← ruta_inicial, o la del vecino más cercano desde inicio si no se dio ninguna
    repetir:
        para cada par de posiciones i < j (no contiguas):
            Δ ← coste(ruta[i]→ruta[j]) + coste(ruta[i+1]→ruta[j+1])
                − coste(ruta[i]→ruta[i+1]) − coste(ruta[j]→ruta[j+1])
                + (coste del tramo i+1..j al revés − coste del tramo i+1..j hacia delante)
        (i*, j*) ← el par con menor Δ
        si Δ(i*, j*) < 0:  invertir el tramo ruta[i*+1 .. j*]
        si no:             parar                            # óptimo local
    devolver (ruta, coste de la ruta)
</pre>

**Implementación.** En cada vuelta se calcula la variación `Δ` de **todos** los pares `(i, j)` a la vez con `numpy` (matrices `n×n`) y se aplica el **mejor** movimiento (el de menor `Δ`). Es la variante "mejor mejora", más fácil de explicar que mirar los pares uno a uno y de resultado más estable. Con `max_iteraciones=0` la función devuelve directamente la ruta de partida, lo que permite medir cuánto ha mejorado el 2-opt.

**Complejidad.** **Tiempo:** cada vuelta cuesta **O(n²)** (todos los pares); el número de vueltas depende de la ruta de partida, y en la práctica es del orden de `n` (una mejora por cruce), así que el conjunto suele rondar **O(n³)**. No hay una cota polinómica en el peor caso. **Memoria: O(n²)**, por las matrices de costes y de variaciones.

**Propiedades.**

- **Tipo:** **heurístico de mejora local** (necesita una ruta de partida; aquí, el vecino más cercano).
- **¿Siempre encuentra el óptimo?** **No**: se detiene en el primer óptimo local, que puede no ser el global. Cuanto mejor sea la ruta de partida, mejor suele ser el resultado.
- **¿Determinista?** **Sí**, dada la ruta de partida (los empates se resuelven siempre igual). Si se parte de rutas aleatorias, distintos arranques acaban en óptimos locales distintos.
- **¿Garantía de calidad?** **Ninguna útil** para un grafo cualquiera. En la práctica, sobre instancias de carreteras suele quedar a unos pocos puntos porcentuales del óptimo, y bastante mejor que el vecino más cercano de partida (se mide abajo).
- **¿Necesita simetría?** **No**: se comprueba el coste de cada arista en su sentido, así que sirve para `nx.DiGraph` y `nx.Graph`.
- **¿Hasta cuántas ciudades?** **Todas las del trabajo** (47 y 75 tardan milisegundos) y bastantes más: el límite lo pone el `O(n²)` de memoria por vuelta, no el tiempo.
- **Uso en el trabajo:** es la mejora natural del vecino más cercano y el punto de comparación "barato y bueno" para las metaheurísticas (recocido simulado, genético).

</div>

**💻 CÓDIGO**

La función arranca con la ruta del vecino más cercano (o con la que se le pase en `ruta_inicial`) y aplica 2-opt hasta no poder mejorar. La matriz `delta` guarda la variación de coste de cada movimiento posible (`i`, `j`); los movimientos no válidos —`i ≥ j`, o el par `(0, n−1)`, cuyas aristas son contiguas— se marcan con infinito. La ciudad de inicio queda siempre en la primera posición, porque el tramo que se invierte empieza en `i+1 ≥ 1`.

In [8]:
def dos_opt(K, inicio=None, ruta_inicial=None, max_iteraciones=100000):
    # 2-opt (mejora local) para el problema del viajante: parte de una ruta (por defecto la del vecino
    # más cercano) y mientras pueda, deshace dos aristas que se cruzan invirtiendo el tramo entre ellas.
    # Contrato: K es el grafo de ciudades (completo, K[a][b]["peso"]) y devuelve (ruta, coste), con la
    # ruta completa (cada ciudad una vez, el viajante vuelve después a la primera) y el coste del ciclo
    # incluida la vuelta. Sirve para K dirigido y no dirigido: en un grafo dirigido, invertir un tramo
    # cambia el coste de sus aristas interiores, y aquí se tiene en cuenta.

    ciudades = list(K.nodes)
    n = len(ciudades)

    # Casos triviales y comprobaciones
    if n == 0:
        return [], 0.0
    if inicio is None:
        inicio = ciudades[0]
    elif inicio not in K:
        raise KeyError(f"La ciudad de inicio {inicio!r} no está en el grafo")
    orden = [inicio] + [c for c in ciudades if c != inicio]   # índice 0 = inicio
    posicion = {c: i for i, c in enumerate(orden)}
    if n <= 2:                                                # con 1 o 2 ciudades no hay nada que mejorar
        return orden, (float(K[orden[0]][orden[1]]["peso"] + K[orden[1]][orden[0]]["peso"]) if n == 2 else 0.0)

    # Matriz de costes W[i][j] (rápida de consultar)
    W = np.array([[0.0 if a == b else K[a][b]["peso"] for b in orden] for a in orden])

    # Ruta de partida (como lista de índices): la dada, o el vecino más cercano desde el inicio
    if ruta_inicial is not None:
        t = np.array([posicion[c] for c in ruta_inicial])
        t = np.roll(t, -int(np.where(t == 0)[0][0]))        # la rotamos para que empiece en el inicio
    else:
        pendientes = set(range(1, n))
        t = [0]
        while pendientes:
            sig = min(pendientes, key=lambda c: (W[t[-1], c], c))
            t.append(sig)
            pendientes.remove(sig)
        t = np.array(t)

    # Posiciones de la ruta: i < j. Quitar las aristas (t[i], t[i+1]) y (t[j], t[j+1]) y unir
    # t[i]→t[j] y t[i+1]→t[j+1] equivale a invertir el tramo t[i+1..j].
    I, J = np.meshgrid(np.arange(n), np.arange(n), indexing="ij")
    valido = (I < J) & ~((I == 0) & (J == n - 1))            # (0, n-1) son aristas contiguas: no vale

    for _ in range(max_iteraciones):
        # Sumas acumuladas del coste de recorrer la ruta hacia delante (F) y hacia atrás (B):
        # sirven para saber en O(1) cuánto cuesta un tramo en un sentido y en el contrario
        F = np.concatenate(([0.0], np.cumsum(W[t[:-1], t[1:]])))
        B = np.concatenate(([0.0], np.cumsum(W[t[1:], t[:-1]])))
        a, b = t[I], t[(I + 1) % n]                          # extremos de la primera arista
        c, d = t[J], t[(J + 1) % n]                          # extremos de la segunda arista
        # Variación del coste al aplicar el movimiento (negativa = mejora). Para el tramo
        # interior, se cambia su coste hacia delante por su coste recorrido al revés.
        delta = (W[a, c] + W[b, d] - W[a, b] - W[c, d]
                 + (B[J] - B[np.minimum(I + 1, n - 1)]) - (F[J] - F[np.minimum(I + 1, n - 1)]))
        delta = np.where(valido, delta, np.inf)

        # Aplicamos el mejor movimiento posible; si ninguno mejora, hemos llegado a un óptimo local
        i, j = np.unravel_index(np.argmin(delta), delta.shape)
        if delta[i, j] > -1e-9:
            break
        t[i + 1:j + 1] = t[i + 1:j + 1][::-1].copy()

    ruta = [orden[k] for k in t]
    coste = float(sum(W[t[k], t[(k + 1) % n]] for k in range(n)))
    return ruta, coste

**🧪 PRUEBAS**

**Prueba rápida.** Tres comprobaciones:

1. **Versión pequeña real:** las 6 ciudades de Galicia empezando en Vigo, con `validar_ruta` y `medir` (tiempo y memoria).
2. **Cuánto mejora:** con `max_iteraciones=0` la función devuelve la ruta de partida, así que se compara el vecino más cercano con el resultado del 2-opt.
3. **Referencia exacta:** el mejor de los `6! = 720` ciclos posibles.

In [9]:
# ── 1. Versión pequeña real: Galicia (6 ciudades), empezando en Vigo ──
K = cargar_version(aristas="completo", nodos="capitales_y_100k", ambito="galicia", peso="km", general="grafo_dirigido")
ruta, coste = dos_opt(K, inicio="Vigo")
validar_ruta(K, ruta)                               # comprueba que la ruta visita cada ciudad una vez
print(" → ".join(ruta), f"| coste: {coste:.1f} {K.graph['unidad']}")
print(medir(dos_opt, K, memoria=True, inicio="Vigo"))

# ── 2. Ruta de partida (vecino más cercano) y mejora conseguida ──
_, coste_inicial = dos_opt(K, inicio="Vigo", max_iteraciones=0)
print(f"\nvecino más cercano: {coste_inicial:.3f} km → 2-opt: {coste:.3f} km")

# ── 3. Referencia: el mejor de los 6! = 720 ciclos ──
optimo = min(coste_ruta(K, list(p)) for p in itertools.permutations(K.nodes))
print(f"óptimo de los 720 ciclos: {optimo:.3f} km | 2-opt: {coste:.3f} km | diferencia: {coste - optimo:+.3f} km")

Vigo → Pontevedra → Santiago de Compostela → A Coruña → Lugo → Ourense | coste: 465.8 km
{'algoritmo': 'dos_opt', 'ciudades': 6, 'coste': 465.817, 'unidad': 'km', 'segundos': 0.0005, 'memoria_mb': 0.01}

vecino más cercano: 465.817 km → 2-opt: 465.817 km
óptimo de los 720 ciclos: 460.438 km | 2-opt: 465.817 km | diferencia: +5.379 km


**Calidad frente al óptimo.** Sobre subgrafos de Andalucía (de 5 a 9 ciudades) se compara el 2-opt con el óptimo (enumerando todos los ciclos), ejecutándolo **desde cada una de sus ciudades** para ver el mejor y el peor caso, ya que el punto de partida del vecino más cercano cambia con el inicio.

In [10]:
# ── 2-opt frente al óptimo en instancias cada vez mayores (Andalucía, grafo dirigido) ──
KAnd = cargar_version(aristas="completo", nodos="capitales_y_100k", ambito="andalucia", peso="km", general="grafo_dirigido")
print(f"{'ciudades':>8} | {'óptimo':>9} | {'mejor 2-opt':>11} | {'peor 2-opt':>10} | {'gap del peor':>12}")
for n in range(5, 10):
    Kn = KAnd.subgraph(list(KAnd.nodes)[:n]).copy()
    optimo = min(coste_ruta(Kn, list(p)) for p in itertools.permutations(Kn.nodes))
    costes = [dos_opt(Kn, inicio=c)[1] for c in Kn.nodes]
    print(f"{n:>8} | {optimo:>9.1f} | {min(costes):>11.1f} | {max(costes):>10.1f} | {(max(costes) - optimo) / optimo * 100:>+11.1f}%")

ciudades |    óptimo | mejor 2-opt | peor 2-opt | gap del peor
       5 |     857.1 |       857.1 |      858.8 |        +0.2%
       6 |    1038.2 |      1038.2 |     1039.3 |        +0.1%
       7 |    1072.2 |      1072.2 |     1073.9 |        +0.2%


       8 |    1220.7 |      1220.7 |     1223.1 |        +0.2%


       9 |    1226.4 |      1226.4 |     1226.7 |        +0.0%


**Efecto de la ruta de partida (47 capitales).** Se compara el vecino más cercano, el 2-opt partiendo de él y el 2-opt partiendo de **10 rutas aleatorias** (semilla fija `42`). Como referencia, el óptimo de esta versión (`completo`/`km`/dirigido) es **5.917,562 km**, calculado con el algoritmo DFJ del notebook de Jiahao.

In [11]:
# ── Efecto de la ruta de partida sobre 47 capitales (dirigido, km) ──
K47 = cargar_version(aristas="completo", nodos="capitales", ambito="espana", peso="km", general="grafo_dirigido")
OPTIMO_47 = 5917.562                                 # óptimo calculado con DFJ (notebook de Jiahao)
_, nn = dos_opt(K47, max_iteraciones=0)
_, desde_nn = dos_opt(K47)
rng = np.random.default_rng(42)                      # semilla fija: los resultados se pueden repetir
ciudades47 = list(K47.nodes)
desde_azar = [dos_opt(K47, ruta_inicial=list(rng.permutation(ciudades47)))[1] for _ in range(10)]

def linea(nombre, valor):
    print(f"{nombre:<40} {valor:>9.1f} km   ({(valor - OPTIMO_47) / OPTIMO_47 * 100:>+5.1f} % sobre el óptimo)")
linea("vecino más cercano (ruta de partida)", nn)
linea("2-opt desde el vecino más cercano", desde_nn)
linea("2-opt desde azar: mejor de 10", min(desde_azar))
linea("2-opt desde azar: media de 10", float(np.mean(desde_azar)))
linea("2-opt desde azar: peor de 10", max(desde_azar))

vecino más cercano (ruta de partida)        6772.1 km   (+14.4 % sobre el óptimo)
2-opt desde el vecino más cercano           6326.0 km   ( +6.9 % sobre el óptimo)
2-opt desde azar: mejor de 10               6122.8 km   ( +3.5 % sobre el óptimo)
2-opt desde azar: media de 10               6273.0 km   ( +6.0 % sobre el óptimo)
2-opt desde azar: peor de 10                6434.1 km   ( +8.7 % sobre el óptimo)


**Escalabilidad.** El 2-opt se mide en todas las versiones de España (4 tipos de carreteras × 2 conjuntos de ciudades × dirigido/no dirigido), partiendo siempre del vecino más cercano desde Madrid o, si no está, desde la primera ciudad. `solo_autovias` tiene menos ciudades porque algunas quedan a más de 10 km de la red.

In [12]:
# ── Escalabilidad: todas las versiones de España, con 47 y 75 ciudades ──
print(f"{'aristas':>21} | {'ciudades':>8} | {'general':>24} | {'vecino cercano':>14} | {'2-opt':>9} | {'mejora':>7} | {'tiempo':>8}")
for aristas in ("completo", "autovias_principales", "sin_autovias", "solo_autovias"):
    for nodos in ("capitales", "capitales_y_100k"):
        for general in ("grafo_dirigido", "grafo_simple_no_dirigido"):
            Kv = cargar_version(aristas=aristas, nodos=nodos, ambito="espana", peso="km", general=general)
            inicio = "Madrid" if "Madrid" in Kv else None
            _, c0 = dos_opt(Kv, inicio=inicio, max_iteraciones=0)
            r = medir(dos_opt, Kv, inicio=inicio)
            print(f"{aristas:>21} | {r['ciudades']:>8} | {general:>24} | {c0:>14.1f} | {r['coste']:>9.1f} | "
                  f"{(c0 - r['coste']) / c0 * 100:>6.1f}% | {r['segundos']:>7.3f}s")

              aristas | ciudades |                  general | vecino cercano |     2-opt |  mejora |   tiempo
             completo |       47 |           grafo_dirigido |         6772.1 |    6326.0 |    6.6% |   0.003s
             completo |       47 | grafo_simple_no_dirigido |         6731.8 |    6267.3 |    6.9% |   0.002s
             completo |       75 |           grafo_dirigido |         8101.9 |    6424.9 |   20.7% |   0.008s
             completo |       75 | grafo_simple_no_dirigido |         7997.6 |    6342.0 |   20.7% |   0.007s
 autovias_principales |       47 |           grafo_dirigido |         7378.6 |    6540.2 |   11.4% |   0.003s
 autovias_principales |       47 | grafo_simple_no_dirigido |         7272.7 |    6433.9 |   11.5% |   0.003s


 autovias_principales |       75 |           grafo_dirigido |         8385.8 |    6788.9 |   19.0% |   0.009s
 autovias_principales |       75 | grafo_simple_no_dirigido |         8234.8 |    6634.4 |   19.4% |   0.007s
         sin_autovias |       47 |           grafo_dirigido |        10913.8 |    7973.7 |   26.9% |   0.011s
         sin_autovias |       47 | grafo_simple_no_dirigido |         9650.1 |    7590.8 |   21.3% |   0.004s
         sin_autovias |       75 |           grafo_dirigido |        11080.5 |    8660.2 |   21.8% |   0.009s
         sin_autovias |       75 | grafo_simple_no_dirigido |        10302.2 |    8504.8 |   17.4% |   0.009s
        solo_autovias |       44 |           grafo_dirigido |         9901.2 |    8783.9 |   11.3% |   0.007s


        solo_autovias |       44 | grafo_simple_no_dirigido |         8728.8 |    6989.3 |   19.9% |   0.006s
        solo_autovias |       72 |           grafo_dirigido |        10168.5 |   10055.0 |    1.1% |   0.004s
        solo_autovias |       72 | grafo_simple_no_dirigido |         8810.9 |    7329.9 |   16.8% |   0.008s


<div style="border-left:5px solid #55A868;color:#000;background:#EAF6ED;padding:10px 14px;margin:10px 0;border-radius:4px;">

**🧮 RESULTADOS**

**Prueba rápida (Galicia, 6 ciudades, desde Vigo).** El 2-opt parte del vecino más cercano (465,8 km) y **no encuentra mejora**: ese recorrido ya es un óptimo local, aunque el óptimo real es **460,4 km** (+5,4 km, +1,2 %). Es un ejemplo claro de la limitación de la búsqueda local: se queda atascada en el primer óptimo local.

**Calidad frente al óptimo** (Andalucía, 5 a 9 ciudades, dirigido, ejecutando desde cada ciudad): el 2-opt **alcanza el óptimo en el mejor caso en las 5 instancias** y en el peor caso se queda a un **0,0–0,2 %**. Con instancias tan pequeñas el vecino más cercano de partida ya es muy bueno.

**Efecto de la ruta de partida** (47 capitales, `completo`/`km`/dirigido; el óptimo es 5.917,6 km según DFJ):

| Método | Coste | Sobre el óptimo |
|---|---:|---:|
| Vecino más cercano | 6.772,1 km | +14,4 % |
| 2-opt desde el vecino más cercano | 6.326,0 km | **+6,9 %** |
| 2-opt desde azar: mejor de 10 | 6.122,8 km | +3,5 % |
| 2-opt desde azar: media de 10 | 6.273,0 km | +6,0 % |
| 2-opt desde azar: peor de 10 | 6.434,1 km | +8,7 % |

El 2-opt **reduce a la mitad el error del vecino más cercano** (de +14,4 % a +6,9 %). Curiosamente, partir de rutas aleatorias da de media un resultado parecido o algo mejor (+6,0 %) y el mejor de 10 arranques llega a +3,5 %: un buen punto de partida no garantiza un mejor óptimo local, y **repetir con varios arranques** es una mejora barata (cada ejecución tarda milisegundos).

**Escalabilidad** (todas las versiones de España, km): tarda **menos de 10 ms** incluso con 75 ciudades. Mejora al vecino más cercano entre un **1 % y un 27 %** según la versión; las mayores mejoras se dan con 75 ciudades y en la red `sin_autovias` (hasta 26,9 % en las 47 capitales dirigidas), donde el vecino más cercano comete más errores. Memoria: `O(n²)`, irrelevante con estos tamaños.

</div>

---